In [7]:
import os
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
from pathlib import Path
PHARAOH_FOLDER = Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\raw\pharaohs_docs")
LANDMARK_FOLDER = Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\raw\landmarks_docs")

In [3]:
!nvidia-smi

Sat Feb 28 15:10:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   49C    P8              3W /   50W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
qwen_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", tokenizer_kwargs={"padding_side": "left"})

#mxbai_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", ","],
    length_function=len,
    is_separator_regex=False,
)

In [ ]:
import numpy as np

def generate_embeddings_and_save(
    data_path,
    persist_path,
    collection_name,
    embedding_model,
    mrl_dim: int = None 
):
    loader = DirectoryLoader(
        data_path,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    documents = loader.load()

    for doc in documents:
        file_name = os.path.basename(doc.metadata["source"])
        doc.metadata = {"entity_name": file_name}

    split_docs = text_splitter.split_documents(documents)

    import chromadb
    client = chromadb.PersistentClient(path=persist_path)

    collection = client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    texts = [doc.page_content for doc in split_docs]
    metadatas = [doc.metadata for doc in split_docs]
    ids = [f"{meta['entity_name']}_{i}" for i, meta in enumerate(metadatas)]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True 
    )

    if mrl_dim is not None:
        embeddings = embeddings[:, :mrl_dim]
        embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas,
        ids=ids
    )

In [ ]:
# Pharaohs - Qwen
generate_embeddings_and_save(
    data_path=PHARAOH_FOLDER,
    persist_path=Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\embeddings\pharaohs_qwen_db"),
    collection_name="pharaohs",
    embedding_model=qwen_model,
    mrl_dim=None
)

In [ ]:

# Landmarks - Qwen
generate_embeddings_and_save(
    data_path=LANDMARK_FOLDER,
    persist_path=Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\embeddings\landmarks_qwen_db"),
    collection_name="landmarks",
    embedding_model=qwen_model,
    mrl_dim=None
)

In [ ]:
client = chromadb.PersistentClient(path=Path(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\embeddings\pharohs_qwen_db"))

collection = client.get_collection("pharaohs")

query = "Tell me about Queen Tiye, wife of Amenhotep III"
query_embedding = qwen_model.encode([query], normalize_embeddings=True)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

for meta in results["metadatas"][0]:
    print("*" * 50)
    print(meta['entity_name'])
    